# AUV Trajectory Evaluation

This notebook selects a trained profile from `simulation.isaac.ppo.architectures`, evaluates its Actor only, and visualizes saved metrics. The Critic is neither instantiated nor executed. All helper functions live in `simulation/isaac/trajectory/experiment.py`.

In [ ]:
from pathlib import Path
from dataclasses import replace
import importlib
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "simulation/isaac").is_dir():
    raise RuntimeError("请从项目根目录启动 eval.ipynb。")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
from IPython.display import display
# Reload the registry before workflow helpers so a running kernel sees newly
# added architecture profiles without requiring a restart.
importlib.invalidate_caches()
import simulation.isaac.ppo.architectures.registry as architecture_registry
import simulation.isaac.ppo.architectures as mlp_architectures
importlib.reload(architecture_registry)
importlib.reload(mlp_architectures)
import simulation.isaac.trajectory.experiment as experiment_tools
# Jupyter caches imported modules. Reload so this cell always uses the current dataclass/API definitions.
importlib.reload(experiment_tools)
from simulation.isaac.trajectory.experiment import (
    EvalRequest, ExperimentSpec, best_by_trajectory, collect_summary_df, eval_request_case_label,
    configure_plots, load_eval_log, plot_checkpoint_gallery, plot_eval_detail,
    plot_rmse_heatmap, plot_rmse_summary, quick_numeric_report, resolve_detail_checkpoint,
    resolve_detail_trajectory, resolve_run, run_eval_matrix, runs_dataframe,
    save_summary_table, show_active_paths,
)
print(f"Experiment tools: {experiment_tools.__file__}")

configure_plots()
# Must match the architecture selected by simulation.isaac.train.
MLP_ARCHITECTURE = "mlp_history_5"
ISAACLAB_ROOT = Path("/home/jining_yang/IsaacLab")
RLPOLICY_ROOT = REPO_ROOT / "simulation/isaac/rlpolicy"
SPEC = ExperimentSpec(
    isaaclab_root=ISAACLAB_ROOT, rlpolicy_root=RLPOLICY_ROOT, mlp_architecture=MLP_ARCHITECTURE
)

# Must match the policy stored in the training run's params/env.yaml (policy_0 ... policy_6).
REWARD_PROFILE = "policy_6"
EVAL_SEED = 42
# Use the deterministic modeled pool profile selected for training. Set None only to benchmark
# against the built-in nominal pool instead.
EVAL_ENVIRONMENT_PROFILE = REPO_ROOT / "environment/hydrodynamics/coefficients/auv_pool_openfoam_hydrodynamics_v1.json"
# Default to the same versioned sensor/actuator DR recipe used in training.
# Set SAMPLE_DOMAIN_RANDOMIZATION=False only for the explicitly nominal report.
EVAL_DOMAIN_RANDOMIZATION_SPEC = REPO_ROOT / "environment/profiles/configs/auv_pool_openfoam_hydrodynamics_dr_v1.json"
SAMPLE_DOMAIN_RANDOMIZATION = True
# 直接指定 simulation.isaac.train 生成的完整运行目录；不再隐式搜索外部 IsaacLab logs。
POLICY_RUN_DIR = None  # 例如：SPEC.logs_root / "2026-08-15_12-00-00_t60_policy_6"
FINAL_CHECKPOINT = "latest"
EVAL_CHECKPOINTS = FINAL_CHECKPOINT
# This runs the two final gate reports below (nominal and full Stage-4 DR).
RUN_FINAL_ACCEPTANCE = False
# Keep the broader six-trajectory generalization matrix separate from the final gate.
RUN_EVAL = False
# Fixed-current diagnostics reuse the same seed and random_smooth curves; enable only after
# the resumed DR curriculum is running. The ordinary EVAL above remains the full-DR comparison.
RUN_CURRENT_SWEEP = False
CURRENT_SWEEP_MAGNITUDES_MPS = (0.0, 0.05, 0.10, 0.15, 0.20)
CURRENT_SWEEP_DIRECTION_W = (1.0, 0.0, 0.0)
SAVE_FIGURES = True
# Keep VS Code's Jupyter WebView light: figures are saved as PNG and closed by default.
# Set True temporarily only when an inline figure is needed.
SHOW_INLINE_PLOTS = True
DETAIL_TRAJECTORY = "random_smooth"
# The detailed plot and gallery use the newest selected checkpoint.
DETAIL_CHECKPOINT = "latest"
DETAIL_ENV_ID = 0

EVAL = EvalRequest(
    reward_profile=REWARD_PROFILE,
    seed=EVAL_SEED,
    checkpoint=EVAL_CHECKPOINTS,
    trajectories=("lissajous", "helix", "spiral", "chirp", "racetrack", "random_smooth"),
    duration_s=32.0,
    headless=True,
    # Recreate summaries so no result from an earlier evaluation configuration is reused.
    skip_existing=False,
    align_initial_target=True,
    random_curve_count=8,
    trajectory_amp_x_range=(0.60, 0.78), trajectory_amp_y_range=(0.55, 0.75),
    trajectory_amp_z_range=(0.08, 0.20), trajectory_period_range=(10.0, 20.0),
    environment_profile=EVAL_ENVIRONMENT_PROFILE,
    domain_randomization_spec=EVAL_DOMAIN_RANDOMIZATION_SPEC,
    sample_domain_randomization=SAMPLE_DOMAIN_RANDOMIZATION,
    # A fresh evaluator starts at local DR stage 0; final robustness must explicitly request stage 4.
    eval_disturbance_stage=4,
    evaluation_label="t60_final_generalization_robust_stage4",
)
EVAL_CASE_LABEL = eval_request_case_label(EVAL)

# These requests reproduce the final curriculum gate distribution (stage 3 trajectory difficulty)
# with 32 held-out random_smooth curves. Re-timing may lengthen individual periods;
# use each summary's effective period and measured path length instead of assuming four laps.
FINAL_ACCEPTANCE_COMMON = dict(
    reward_profile=REWARD_PROFILE, checkpoint=FINAL_CHECKPOINT, trajectories=("random_smooth",),
    duration_s=40.0, headless=True, skip_existing=False, align_initial_target=True,
    random_curve_count=32, num_envs=32, trajectory_amp_x_range=(0.60, 0.78),
    trajectory_amp_y_range=(0.55, 0.75), trajectory_amp_z_range=(0.08, 0.20),
    trajectory_period_range=(10.0, 20.0),
    environment_profile=EVAL_ENVIRONMENT_PROFILE,
    domain_randomization_spec=EVAL_DOMAIN_RANDOMIZATION_SPEC, keep_boundaries=True,
)
FINAL_ACCEPTANCE_EVALS = (
    EvalRequest(seed=10173, sample_domain_randomization=False, eval_disturbance_stage=-1,
                evaluation_label="t60_final_nominal", **FINAL_ACCEPTANCE_COMMON),
    EvalRequest(seed=20971, sample_domain_randomization=True, eval_disturbance_stage=4,
                evaluation_label="t60_final_robust_stage4", **FINAL_ACCEPTANCE_COMMON),
)
FINAL_GATE_LIMITS = {
    "t60_final_nominal": {"position_error_p95": 1.05, "velocity_rmse": 0.65},
    "t60_final_robust_stage4": {"position_error_p95": 1.25, "velocity_rmse": 0.75},
}
CURRENT_SWEEP_BASE = replace(
    EVAL, trajectories=("random_smooth",), duration_s=20.0, align_initial_target=True,
    random_curve_count=32, num_envs=32, sample_domain_randomization=False,
    eval_disturbance_stage=-1, skip_existing=False,
)
CURRENT_SWEEP_REQUESTS = tuple(
    replace(
        CURRENT_SWEEP_BASE,
        evaluation_label=f"t60_current_only_{magnitude:.3f}mps",
        disturbance_name=f"t60_current_only_{magnitude:.3f}mps",
        eval_current=tuple(magnitude * component for component in CURRENT_SWEEP_DIRECTION_W),
    )
    for magnitude in CURRENT_SWEEP_MAGNITUDES_MPS
)

def finish_figure(figure):
    """Optionally render one figure, then always release its WebView/GPU memory."""
    if SHOW_INLINE_PLOTS:
        display(figure)
    plt.close(figure)

## 选择一个明确的策略运行目录

In [ ]:
display(runs_dataframe(SPEC, reward_profile=REWARD_PROFILE).head(20))
if POLICY_RUN_DIR is None:
    ACTIVE_RUN = None
    print("请把 POLICY_RUN_DIR 设为 simulation.isaac.train 生成的运行目录。")
else:
    selected_dir = Path(POLICY_RUN_DIR).expanduser().resolve()
    expected_dir = (SPEC.logs_root / selected_dir.name).resolve()
    if selected_dir != expected_dir:
        raise ValueError(f"运行目录必须位于 {SPEC.logs_root}，收到 {selected_dir}")
    ACTIVE_RUN = resolve_run(SPEC, selected_dir.name, REWARD_PROFILE)
    show_active_paths(SPEC, ACTIVE_RUN, REWARD_PROFILE)

## Final acceptance gate

The final policy is pinned to `policy_6` / `model_480.pt`. It must pass both matched held-out reports: nominal and the full Stage-4 domain-randomization recipe. The robust request explicitly sets `eval_disturbance_stage=4`; otherwise a fresh evaluator would only test Stage 0.

In [ ]:
if ACTIVE_RUN is not None and RUN_FINAL_ACCEPTANCE:
    final_gate_rows = []
    for final_eval in FINAL_ACCEPTANCE_EVALS:
        run_eval_matrix(SPEC, final_eval, load_run=ACTIVE_RUN, execute=True)
        case_label = eval_request_case_label(final_eval)
        final_summary = collect_summary_df(
            SPEC, ACTIVE_RUN, case_label=case_label
        )
        final_summary = final_summary[final_summary["checkpoint_name"] == FINAL_CHECKPOINT].copy()
        display(final_summary)
        if final_summary.empty:
            print(f"{case_label}: no final summary was produced.")
            continue
        row = final_summary.iloc[0]
        limits = FINAL_GATE_LIMITS[case_label]
        passed = (
            row["position_error_p95"] <= limits["position_error_p95"]
            and row["velocity_rmse"] <= limits["velocity_rmse"]
            and row["any_failure_rate"] <= 0.02
            and row["reference_valid"] >= 1
            and row["reference_within_kinematic_envelope"] >= 1
        )
        final_gate_rows.append({"case": case_label, "passed": passed, **row.to_dict()})
        print(f"{case_label}: {'PASS' if passed else 'FAIL'} | "
              f"p95={row['position_error_p95']:.3f}/{limits['position_error_p95']:.2f} m, "
              f"vel={row['velocity_rmse']:.3f}/{limits['velocity_rmse']:.2f} m/s, "
              f"any_failure={row['any_failure_rate']:.3%}/2.000%, "
              f"reference_valid={int(row['reference_valid'])}, "
              f"path_min={row['min_reference_path_length_m']:.3f} m, "
              f"period_eff={row['effective_period_mean_s']:.2f} s")
    if len(final_gate_rows) == len(FINAL_ACCEPTANCE_EVALS):
        print('FINAL ACCEPTANCE:', 'PASS' if all(row['passed'] for row in final_gate_rows) else 'FAIL')
elif ACTIVE_RUN is not None:
    print('RUN_FINAL_ACCEPTANCE is False; final evaluations were not launched.')

## Optional six-trajectory robust generalization matrix

This is diagnostic only; it is not the final acceptance gate above. It uses the same full Stage-4 DR recipe.

In [ ]:
if ACTIVE_RUN is not None:
    _, planned_commands = run_eval_matrix(SPEC, EVAL, load_run=ACTIVE_RUN, execute=RUN_EVAL)
    if not RUN_EVAL:
        print(f"RUN_EVAL is False; {len(planned_commands)} generalization commands were previewed only.")

## Fixed-current sweep

These five matched `random_smooth` evaluations hold seed and curve set fixed while changing only a world-frame current. Compare them with the ordinary full-DR evaluation above; inspect `requested_to_applied_action_rms` and realized-thrust fields in the resulting CSVs.

In [ ]:
if ACTIVE_RUN is not None and RUN_CURRENT_SWEEP:
    for current_eval in CURRENT_SWEEP_REQUESTS:
        run_eval_matrix(SPEC, current_eval, load_run=ACTIVE_RUN, execute=True)
elif not RUN_CURRENT_SWEEP:
    print("RUN_CURRENT_SWEEP is False; fixed-current commands are configured but not launched.")

## Collect metrics

In [ ]:
summary_df = (
    collect_summary_df(SPEC, ACTIVE_RUN, case_label=EVAL_CASE_LABEL) if ACTIVE_RUN else None
)
# A run may contain older evaluation files. Keep an explicit checkpoint comparison narrow;
# the 'latest'/'all' selectors already resolve from the actual run.
if summary_df is not None and not summary_df.empty and not isinstance(EVAL_CHECKPOINTS, str):
    summary_df = summary_df[summary_df["checkpoint_name"].isin(EVAL_CHECKPOINTS)].copy()
if summary_df is None or summary_df.empty:
    print("No evaluated summaries found.")
else:
    save_summary_table(SPEC, ACTIVE_RUN, summary_df, case_label=EVAL_CASE_LABEL)
    columns = [
        "reward_profile", "trajectory", "checkpoint_name",
        "position_rmse", "velocity_rmse",
        "mean_command_heading_error_deg", "mean_motion_sideslip_error_deg",
        "mean_action_rms", "mean_action_rate_rms", "mean_reward_per_step",
    ]
    display(summary_df[[column for column in columns if column in summary_df.columns]])
    display(best_by_trajectory(summary_df))
    display(quick_numeric_report(summary_df))

## Aggregate plots

In [ ]:
if summary_df is not None and not summary_df.empty:
    rmse_figure, _ = plot_rmse_summary(
        SPEC, ACTIVE_RUN, summary_df, case_label=EVAL_CASE_LABEL, save=SAVE_FIGURES
    )
    finish_figure(rmse_figure)
    heatmap_figure, _, rmse_pivot = plot_rmse_heatmap(
        SPEC, ACTIVE_RUN, summary_df, case_label=EVAL_CASE_LABEL, save=SAVE_FIGURES
    )
    finish_figure(heatmap_figure)
    display(rmse_pivot)

## Detailed trajectory and gallery

In [ ]:
if summary_df is not None and not summary_df.empty:
    detail_trajectory = resolve_detail_trajectory(summary_df, DETAIL_TRAJECTORY)
    detail_checkpoint = resolve_detail_checkpoint(summary_df, detail_trajectory, DETAIL_CHECKPOINT)
    detail_df, detail_path = load_eval_log(
        SPEC, ACTIVE_RUN, detail_checkpoint, detail_trajectory, EVAL_CASE_LABEL
    )
    print(f"Loaded {len(detail_df)} rows from {detail_path}")
    detail_figure = plot_eval_detail(
        SPEC, ACTIVE_RUN, detail_df, detail_checkpoint, detail_trajectory,
        case_label=EVAL_CASE_LABEL, env_id=DETAIL_ENV_ID, save=SAVE_FIGURES,
    )
    finish_figure(detail_figure)
    gallery_figure = plot_checkpoint_gallery(
        SPEC, ACTIVE_RUN, summary_df, detail_checkpoint,
        case_label=EVAL_CASE_LABEL, save=SAVE_FIGURES
    )
    finish_figure(gallery_figure)